In [10]:
!pip install faker -q

### K-1. Import Library dan Inisialisasi
Langkah ini menyiapkan pustaka utama yang dibutuhkan sepanjang praktikum, yaitu NumPy, pandas, Faker, dan random untuk pembangkitan dataset sintetis serta manipulasi data tabular.

In [11]:
import numpy as np
import pandas as pd
from faker import Faker
import random

### K-2. Membuat Dataset Sintetis (Simulasi Data Acquisition)
Mengatur random seed (SEED = 42) untuk menjamin sifat deterministik/reproducible, kemudian mensimulasikan data mentah transaksi marketplace sebanyak 515 baris yang sengaja disuntikkan variasi format harga, tanggal, kapitalisasi, missing values, dan baris duplikat.

In [12]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [13]:
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
fake = Faker("id_ID")
Faker.seed(SEED)

N = 500
kategori_produk = ["Elektronik", "Fashion", "Kesehatan", "Rumah Tangga", "Olahraga", "Buku"]
metode_bayar = ["Transfer Bank", "E-Wallet", "COD", "Kartu Kredit"]

rows = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake.name()
    produk = fake.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [
        str(harga_dasar),
        f"Rp{harga_dasar:,}".replace(",", "."),
        f"{harga_dasar}.0",
        f" {harga_dasar}",
    ]
    harga = random.choice(harga_variants)

    tgl = fake.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3:
        metode = metode.lower()
    if random.random() < 0.2:
        kategori = kategori.upper() + " "

    kota = fake.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows.append({
        "transaction_id": trx_id,
        "customer_name": nama_pelanggan,
        "product_name": produk.strip(),
        "category": kategori,
        "price": harga,
        "quantity": qty,
        "payment_method": metode,
        "transaction_date": tanggal,
        "shipping_city": kota,
        "rating": rating,
    })

df = pd.DataFrame(rows)

# Suntikkan missing value pada beberapa kolom
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx = df.sample(frac=frac, random_state=SEED).index
    df.loc[idx, col] = np.nan

# Duplikasi 15 baris
dup_rows = df.sample(n=15, random_state=SEED)
df = pd.concat([df, dup_rows], ignore_index=True)
df = df.sample(frac=1, random_state=SEED).reset_index(drop=True)

df.to_csv("transaksi_mentah.csv", index=False)
print("Jumlah baris:", len(df))

Jumlah baris: 515


### K-3. Deteksi dan Penanganan Missing Value
Memeriksa jumlah missing value per kolom. Kolom `customer_name` dan `payment_method` ditangani dengan `dropna()` karena merupakan informasi transaksi inti, `shipping_city` diisi dengan `"Tidak Diketahui"` menggunakan `fillna()`, sedangkan `rating` dibiarkan kosong (NaN) karena bersifat opsional.

In [14]:
print("Missing values per kolom sebelum penanganan:")
print(df.isnull().sum())

# Penanganan missing value sesuai strategi
df = df.dropna(subset=["customer_name", "payment_method"])
df["shipping_city"] = df["shipping_city"].fillna("Tidak Diketahui")

print("\nMissing values per kolom setelah penanganan:")
print(df.isnull().sum())

Missing values per kolom sebelum penanganan:
transaction_id        0
customer_name        20
product_name          0
category              0
price                 0
quantity              0
payment_method       16
transaction_date      0
shipping_city        30
rating              166
dtype: int64

Missing values per kolom setelah penanganan:
transaction_id        0
customer_name         0
product_name          0
category              0
price                 0
quantity              0
payment_method        0
transaction_date      0
shipping_city         0
rating              158
dtype: int64


### K-4. Deteksi dan Penanganan Duplicate
Mendeteksi baris terduplikasi secara keseluruhan dan terduplikasi berdasarkan `transaction_id`, kemudian menghapus duplikat tersebut menggunakan `drop_duplicates()`.

In [15]:
print("Baris duplicate (semua kolom sama):", df.duplicated().sum())
print("transaction_id duplicate:", df['transaction_id'].duplicated().sum())

df = df.drop_duplicates()
print("Jumlah baris setelah drop_duplicates():", len(df))

Baris duplicate (semua kolom sama): 5
transaction_id duplicate: 5
Jumlah baris setelah drop_duplicates(): 490


### K-5. Koreksi Tipe Data dan Standardisasi Format
Melakukan standardisasi kapitalisasi teks kategorikal (Title Case, perbaikan singkatan "COD"), membersihkan kolom harga dari teks/simbol ke tipe numerik float, serta parsing tanggal tercampur (ISO dan non-ISO) secara eksplisit tanpa menggunakan kombinasi berisiko `format="mixed"` dan `dayfirst=True`.

In [16]:
# a. Standardisasi teks kategorikal
for col in ["category", "payment_method", "shipping_city"]:
    df[col] = df[col].astype("string").str.strip().str.title()

df["payment_method"] = df["payment_method"].replace({"Cod": "COD"})

# b. Koreksi tipe data pada kolom price
def bersihkan_harga(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace("Rp", "").replace(".", "").replace(",", ".")
    try:
        return float(x)
    except ValueError:
        return np.nan

df["price"] = df["price"].apply(bersihkan_harga)

# c. Standardisasi format tanggal ke YYYY-MM-DD
def parse_tanggal(x):
    for fmt in ("%Y-%m-%d", "%d/%m/%Y", "%d-%m-%Y"):
        try:
            return pd.to_datetime(x, format=fmt)
        except ValueError:
            continue
    return pd.NaT

df["transaction_date"] = df["transaction_date"].apply(parse_tanggal).dt.strftime("%Y-%m-%d")

# d. Finalisasi tipe data
df["quantity"] = df["quantity"].astype(int)
df["price"] = df["price"].astype(float)

print("Tipe data setelah penyesuaian:")
print(df.dtypes)

Tipe data setelah penyesuaian:
transaction_id              object
customer_name               object
product_name                object
category            string[python]
price                      float64
quantity                     int64
payment_method      string[python]
transaction_date            object
shipping_city       string[python]
rating                     float64
dtype: object


### K-6. Ekspor Dataset Bersih
Mengekspor DataFrame yang telah dibersihkan secara penuh menjadi file `transaksi_bersih.csv` yang akan digunakan pada praktikum selanjutnya.

In [17]:
df.to_csv("transaksi_bersih.csv", index=False)
print("Dataset bersih tersimpan:", len(df), "baris")

Dataset bersih tersimpan: 490 baris


### P. Studi Kasus

#### 1. Penyebab Perbedaan Jumlah Baris (515 vs 490)
Perbedaan 25 baris antara Laporan IT (515 baris mentah) dan Laporan Finance (490 baris bersih) disebabkan oleh dua tahap pra-pemrosesan data:
- **Penghapusan Duplikat (15 baris):** Ditemukan 15 transaksi ganda akibat duplikasi sistem/jaringan yang berhasil dieliminasi.
- **Penghapusan Missing Values Wajib (10 baris):** Ditemukan 10 baris data yang kehilangan informasi penting (nama pembeli atau metode pembayaran) sehingga tidak dapat diidentifikasi secara valid.
Total data yang dibuang adalah 15 + 10 = 25 baris, sehingga 515 - 25 = 490 baris.

#### 2. Keabsahan 490 Baris vs 515 Baris Berdasarkan Konsep Veracity
Angka 490 baris jauh lebih benar dan valid dibandingkan 515 baris jika ditinjau dari dimensi **Veracity** (keakuratan dan derajat kepercayaan data). Menyajikan 515 baris mentah akan menghasilkan laporan keuangan fiktif karena 15 transaksi tercatat ganda (*overstatement*) dan 10 transaksi tidak memiliki kejelasan entitas pembeli/pembayaran. Pembersihan data menaikkan kualitas Veracity sehingga analisis finansial dapat dipercaya.

#### 3. Penjelasan Keputusan Membiarkan Missing Value pada Kolom Rating
Kolom rating bersifat opsional (pembeli tidak diwajibkan memberikan ulasan). Jika missing value diisi (*imputation*) menggunakan nilai tebakan seperti mean atau median, hal ini akan membiaskan distribusi kepuasan pelanggan secara buatan. Fungsi analisis statistik seperti `mean()` pada pandas atau SQL secara otomatis mengabaikan nilai `NaN`, sehingga rating rata-rata yang dihasilkan tetap akurat berdasarkan pembeli yang benar-benar memberikan ulasan.

### Q. Latihan Mandiri (Latihan 1, 2, dan 3)

In [18]:
# --- Latihan 1: Menguji Pipeline dengan SEED = 7 ---
print("--- Latihan 1 (SEED = 7) ---")
SEED_ALT = 7
np.random.seed(SEED_ALT)
random.seed(SEED_ALT)
fake_alt = Faker("id_ID")
Faker.seed(SEED_ALT)

rows_alt = []
for i in range(1, N + 1):
    trx_id = f"TRX{i:05d}"
    nama_pelanggan = fake_alt.name()
    produk = fake_alt.word().capitalize() + " " + random.choice(["Pro", "Lite", "Max", "Basic", ""])
    kategori = random.choice(kategori_produk)
    harga_dasar = random.choice([15000, 25000, 50000, 75000, 120000, 250000, 500000, 1200000])
    qty = random.randint(1, 5)

    harga_variants = [str(harga_dasar), f"Rp{harga_dasar:,}".replace(",", "."), f"{harga_dasar}.0", f" {harga_dasar}"]
    harga = random.choice(harga_variants)

    tgl = fake_alt.date_between(start_date="-90d", end_date="today")
    tgl_variants = [tgl.strftime("%Y-%m-%d"), tgl.strftime("%d/%m/%Y"), tgl.strftime("%d-%m-%Y")]
    tanggal = random.choice(tgl_variants)

    metode = random.choice(metode_bayar)
    if random.random() < 0.3: metode = metode.lower()
    if random.random() < 0.2: kategori = kategori.upper() + " "
    kota = fake_alt.city()
    rating = random.choice([1, 2, 3, 4, 5, None, None])

    rows_alt.append({
        "transaction_id": trx_id, "customer_name": nama_pelanggan, "product_name": produk.strip(),
        "category": kategori, "price": harga, "quantity": qty, "payment_method": metode,
        "transaction_date": tanggal, "shipping_city": kota, "rating": rating,
    })

df_alt = pd.DataFrame(rows_alt)
for col, frac in [("customer_name", 0.02), ("shipping_city", 0.03), ("payment_method", 0.015)]:
    idx_alt = df_alt.sample(frac=frac, random_state=SEED_ALT).index
    df_alt.loc[idx_alt, col] = np.nan

dup_rows_alt = df_alt.sample(n=15, random_state=SEED_ALT)
df_alt = pd.concat([df_alt, dup_rows_alt], ignore_index=True)
df_alt = df_alt.sample(frac=1, random_state=SEED_ALT).reset_index(drop=True)

mentah_alt = len(df_alt)
df_alt = df_alt.dropna(subset=["customer_name", "payment_method"])
df_alt = df_alt.drop_duplicates()
bersih_alt = len(df_alt)

print(f"SEED = 42: Mentah = 515 baris, Bersih = 490 baris")
print(f"SEED = 7  : Mentah = {mentah_alt} baris, Bersih = {bersih_alt} baris")
print("Penjelasan: Jumlah baris mentah tetap 515 karena proporsi duplikat diset fixed (500 + 15). Namun jumlah baris bersih bisa berbeda tergantung baris mana yang terkena missing value acak (karena kombinasi missing pada customer_name dan payment_method bervariasi).")

# --- Latihan 2: Tambahkan kolom is_valid_price ---
print("\n--- Latihan 2 ---")
df["is_valid_price"] = df["price"] > 0
print("Jumlah harga tidak valid (price <= 0 atau NaN):", (~df["is_valid_price"]).sum())
print("Pemeriksaan sampel kolom price dan is_valid_price:")
print(df[["price", "is_valid_price"]].head())

# --- Latihan 3: Hitung Jumlah Transaksi Per Category ---
print("\n--- Latihan 3 ---")
jumlah_per_kategori = df["category"].value_counts()
print("Jumlah transaksi per kategori produk:")
print(jumlah_per_kategori)

--- Latihan 1 (SEED = 7) ---
SEED = 42: Mentah = 515 baris, Bersih = 490 baris
SEED = 7  : Mentah = 515 baris, Bersih = 490 baris
Penjelasan: Jumlah baris mentah tetap 515 karena proporsi duplikat diset fixed (500 + 15). Namun jumlah baris bersih bisa berbeda tergantung baris mana yang terkena missing value acak (karena kombinasi missing pada customer_name dan payment_method bervariasi).

--- Latihan 2 ---
Jumlah harga tidak valid (price <= 0 atau NaN): 0
Pemeriksaan sampel kolom price dan is_valid_price:
      price  is_valid_price
0   50000.0            True
1  500000.0            True
2   25000.0            True
3  250000.0            True
5  250000.0            True

--- Latihan 3 ---
Jumlah transaksi per kategori produk:
category
Olahraga        97
Kesehatan       91
Elektronik      89
Buku            82
Fashion         66
Rumah Tangga    65
Name: count, dtype: Int64


### S. Pertanyaan Evaluasi

#### 1. Perbedaan `dropna()` vs `fillna()` dan Kapan Masing-masing Digunakan
- **`dropna()`:** Menghapus seluruh baris (atau kolom) yang mengandung *missing value* (`NaN`). Digunakan ketika kolom yang kosong merupakan entitas kunci/informasi transaksi inti (seperti `customer_name` atau `payment_method`) yang jika diisi tebakan akan merusak keabsahan data.
- **`fillna()`:** Mengisi *missing value* dengan nilai pengganti tertentu (seperti konstanta, *mean*, *median*, atau *mode*). Digunakan ketika baris data tersebut masih sangat berguna untuk analisis aspek lain meski ada kolom opsional/pelengkap yang kosong (misalnya mengisi `shipping_city` yang kosong dengan nilai `"Tidak Diketahui"`).

#### 2. Alasan `drop_duplicates()` Sebaiknya Dijalankan di Awal Pipeline
Menjalankan `drop_duplicates()` di awal pipeline menghindarkan komputer dari melakukan proses pembersihan yang sia-sia (*overhead processing*) pada baris-baris duplikat yang pada akhirnya akan dibuang. Selain itu, jika duplikasi dihapus setelah proses imputasi atau transformasi format, ada risiko dua transaksi mentah yang berbeda secara tidak sengaja terubah menjadi identik (duplikat palsu).

#### 3. Risiko Perintah `pd.to_datetime(df["transaction_date"], format="mixed", dayfirst=True)`
Kombinasi `format="mixed"` dan `dayfirst=True` berbahaya pada data yang mencakup format ISO (`YYYY-MM-DD`) dan non-ISO (`DD/MM/YYYY`). Parameter `dayfirst=True` akan ikut membalik string tanggal ISO yang sebenarnya sudah tidak ambigu. Sebagai contoh, tanggal ISO `2026-07-11` (11 Juli 2026) dapat salah ditafsirkan oleh pandas menjadi `2026-11-07` (7 November 2026).

#### 4. Alasan Kolom `rating` Dibiarkan Kosong sedangkan `shipping_city` Diisi `"Tidak Diketahui"`
- **Kolom `rating`:** Bersifat opsional (pembeli tidak diwajibkan memberi ulasan). Jika diisi dengan nilai buatan (*imputation*) seperti angka rata-rata, hal tersebut akan mendistorsi/membiaskan analisis kepuasan pelanggan yang sebenarnya. Mengabaikan `NaN` pada kolom rating menjaga keakuratan analisis statistik deskriptif.
- **Kolom `shipping_city`:** Merupakan atribut demografis/logistik. Mengisinya dengan kategori eksplisit `"Tidak Diketahui"` memungkinkan baris transaksi tersebut tetap dihitung dalam analisis volume penjualan tanpa membiaskan data lokasi pengiriman.

#### 5. Keterkaitan Praktikum Ini dengan Dimensi Veracity pada 5V
Praktikum ini berfokus langsung pada dimensi **Veracity** (keakuratan dan derajat kepercayaan data). Data mentah transaksional dari sistem luar sering kali kotor akibat *missing values*, *duplicate records*, dan format yang tidak konsisten. Pra-pemrosesan data (*data preprocessing*) berfungsi menyaring *garbage data* agar dataset memiliki tingkat keandalan yang tinggi sebelum disimpan ke media penyimpanan terstruktur (*storage*) atau dianalisis untuk pengambilan keputusan bisnis.